In [1]:
import os
import pandas as pd

# Define relative path to raw data
RAW_DATA_PATH = os.path.join("..", "data", "raw")

# 1. Load core tables
orders = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_orders_dataset.csv"))
items = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_customers_dataset.csv"))
payments = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_payments_dataset.csv"))

# 2. Filter for completed transactions only
orders_clean = orders[orders["order_status"] == "delivered"].copy()

# 3. Convert timestamp columns to proper datetime format
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col])

# 4. Merge datasets to establish full customer purchase history
merged_df = orders_clean.merge(customers, on="customer_id", how="inner")
merged_df = merged_df.merge(items, on="order_id", how="inner")

print("Merged Data Shape:", merged_df.shape)
print("Unique Customers (by unique ID):", merged_df["customer_unique_id"].nunique())
merged_df.head()

Merged Data Shape: (110197, 18)
Unique Customers (by unique ID): 93358


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [2]:
import pandas as pd
import numpy as np

# 1. Total order item cost
merged_df["total_value"] = merged_df["price"] + merged_df["freight_value"]

# 2. Set snapshot date (1 day after latest order in the dataset)
snapshot_date = merged_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

# 3. Aggregate metrics at customer_unique_id level
rfm = merged_df.groupby("customer_unique_id").agg({
    "order_purchase_timestamp": lambda x: (snapshot_date - x.max()).days, # Recency
    "order_id": "nunique",                                                # Frequency
    "total_value": "sum"                                                  # Monetary
}).reset_index()

rfm.rename(columns={
    "order_purchase_timestamp": "recency_days",
    "order_id": "frequency",
    "total_value": "monetary"
}, inplace=True)

# 4. Score Recency and Monetary (1 to 4 scores)
# Recency: Lower days = higher score (4 is best)
rfm["R_score"] = pd.qcut(rfm["recency_days"], q=4, labels=[4, 3, 2, 1])

# Monetary: Higher spend = higher score (4 is best)
rfm["M_score"] = pd.qcut(rfm["monetary"], q=4, labels=[1, 2, 3, 4])

# Frequency: Most Olist customers purchase 1 time, so use custom bins
rfm["F_score"] = rfm["frequency"].apply(lambda x: 1 if x == 1 else (2 if x == 2 else 3))

# 5. Combine scores into an RFM Segment category
def segment_customer(df):
    if df["F_score"] >= 2 and df["M_score"] >= 3:
        return "Loyal VIP"
    elif df["F_score"] >= 2 and df["R_score"] <= 2:
        return "At Risk / Churn Warning"
    elif df["R_score"] == 4 and df["F_score"] == 1:
        return "New / Recent Customer"
    else:
        return "One-Time Low Value"

rfm["customer_segment"] = rfm.apply(segment_customer, axis=1)

# 6. Save clean dataset and RFM table to data/processed
processed_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

rfm.to_csv(os.path.join(processed_dir, "olist_rfm_summary.csv"), index=False)
merged_df.to_csv(os.path.join(processed_dir, "olist_cleaned_merged.csv"), index=False)

print("Processing Complete!")
print(rfm["customer_segment"].value_counts())

Processing Complete!
customer_segment
One-Time Low Value         68114
New / Recent Customer      22613
Loyal VIP                   2467
At Risk / Churn Warning      164
Name: count, dtype: int64


In [3]:
import sqlite3

# Connect to (or create) SQLite database
db_path = os.path.join("..", "data", "processed", "olist.db")
conn = sqlite3.connect(db_path)

# Save clean DataFrames as tables in the database
merged_df.to_sql("df_orders", conn, if_exists="replace", index=False)
rfm.to_sql("df_rfm", conn, if_exists="replace", index=False)

print("Database created successfully!")
conn.close()

Database created successfully!


In [4]:
import sqlite3
import pandas as pd
import os

# Connect to the SQLite database
db_path = os.path.join("..", "data", "processed", "olist.db")
conn = sqlite3.connect(db_path)

# 1. Define your SQL query
cohort_query = """
WITH customer_first_order AS (
    SELECT 
        customer_unique_id,
        MIN(strftime('%Y-%m-01', order_purchase_timestamp)) AS cohort_month
    FROM df_orders
    GROUP BY customer_unique_id
),
customer_orders AS (
    SELECT 
        o.customer_unique_id,
        c.cohort_month,
        strftime('%Y-%m-01', o.order_purchase_timestamp) AS order_month,
        (CAST(strftime('%Y', o.order_purchase_timestamp) AS INT) - CAST(strftime('%Y', c.cohort_month) AS INT)) * 12 +
        (CAST(strftime('%m', o.order_purchase_timestamp) AS INT) - CAST(strftime('%m', c.cohort_month) AS INT)) AS month_number
    FROM df_orders o
    JOIN customer_first_order c ON o.customer_unique_id = c.customer_unique_id
)
SELECT 
    cohort_month,
    month_number,
    COUNT(DISTINCT customer_unique_id) AS active_customers
FROM customer_orders
GROUP BY cohort_month, month_number
ORDER BY cohort_month, month_number;
"""

# 2. Execute SQL query and load results into a Pandas DataFrame
cohort_df = pd.read_sql_query(cohort_query, conn)

# Always close the database connection when finished
conn.close()

# Display the query results
cohort_df.head(15)

,cohort_month,month_number,active_customers
0,2016-09-01,0,1
1,2016-10-01,0,262
2,2016-10-01,6,1
3,2016-10-01,9,1
4,2016-10-01,11,1
5,2016-10-01,13,1
6,2016-10-01,15,1
7,2016-10-01,17,1
8,2016-10-01,19,2
9,2016-10-01,20,2


In [5]:
import sqlite3
import pandas as pd
import os

# Connect to database
db_path = os.path.join("..", "data", "processed", "olist.db")
conn = sqlite3.connect(db_path)

# Load reviews CSV and push to SQLite
reviews = pd.read_csv(os.path.join("..", "data", "raw", "olist_order_reviews_dataset.csv"))
reviews.to_sql("df_reviews", conn, if_exists="replace", index=False)
conn.close()